### Links and Resources
| Concept | Description |
|---|---|
| [`CREATE FUNCTION` (SQL)](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function.html) | UC SQL UDF DDL |
| [User-defined functions in UC](https://docs.databricks.com/en/udf/unity-catalog.html) | Governance & permissions |
| [Routine privileges](https://docs.databricks.com/en/data-governance/unity-catalog/manage-privileges/privileges.html#functions) | `EXECUTE` grants |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# SQL UDFs in Unity Catalog

A **SQL UDF** is a function whose body is one or more SQL expressions. Once registered in Unity Catalog under a `catalog.schema`, it lives forever (no need to redefine per notebook), is governable with GRANT/REVOKE, and is callable from any SQL or PySpark context that can see it.

Compared to Python UDFs, SQL UDFs:

- **Run inside Catalyst** — no Python worker, no serialization, often as fast as a built-in.
- **Have no NULL trap** — they propagate NULL like every other SQL expression.
- **Are governable** — you grant `EXECUTE` per principal.
- **Are reusable** — define once, share across notebooks/jobs/dashboards/BI tools.

The trade-off: the body must be a SQL expression. No loops, no Python libraries, no model calls.

## Choose a catalog and schema

Set up a workspace where you have `CREATE FUNCTION` privilege. Adjust to your environment.

In [ ]:
CATALOG = "main"          # change to your catalog
SCHEMA  = "default"       # change to your schema

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA  {SCHEMA}")
print("using:", CATALOG, SCHEMA)

## Scalar SQL UDF — single expression

The simplest form: `RETURNS <type> RETURN <expr>`.

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION fmt_money(amount DOUBLE, currency STRING)
RETURNS STRING
RETURN concat(currency, ' ', format_number(amount, 2))
""")

spark.sql("SELECT fmt_money(1234.5, 'EUR') AS s").show()

## With NULL handling and a default

NULL propagates naturally — no Python-style `is None` guard needed.

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION grade(score INT)
RETURNS STRING
RETURN CASE
         WHEN score IS NULL THEN 'unknown'
         WHEN score >= 90  THEN 'A'
         WHEN score >= 75  THEN 'B'
         WHEN score >= 60  THEN 'C'
         ELSE 'F'
       END
""")

spark.sql("SELECT grade(85) AS a, grade(NULL) AS b, grade(50) AS c").show()

## Use it in a query — alongside built-ins

In [ ]:
people = spark.createDataFrame([
    (1, "Alice", 92), (2, "Bob", 73), (3, "Carol", None), (4, "Dana", 55),
], ["id", "name", "score"])
people.createOrReplaceTempView("people")

spark.sql("SELECT id, name, score, grade(score) AS g FROM people").show()

## Functions can call other functions

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION net_amount(gross DOUBLE, vat_rate DOUBLE)
RETURNS DOUBLE
RETURN gross / (1 + vat_rate)
""")

spark.sql("""
CREATE OR REPLACE FUNCTION net_label(gross DOUBLE, vat_rate DOUBLE, currency STRING)
RETURNS STRING
RETURN fmt_money(net_amount(gross, vat_rate), currency)
""")

spark.sql("SELECT net_label(121.0, 0.21, 'EUR') AS net").show()

## Table-valued SQL UDFs

`RETURNS TABLE` lets a function be used as a virtual table. Powerful for parameterized views.

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION top_scorers(min_score INT)
RETURNS TABLE (id INT, name STRING, score INT)
RETURN SELECT id, name, score FROM people WHERE score >= min_score
""")

spark.sql("SELECT * FROM top_scorers(75)").show()

## Discoverability — list & describe

In [ ]:
spark.sql(f"SHOW FUNCTIONS IN {CATALOG}.{SCHEMA} LIKE 'grade'").show(truncate=False)
spark.sql(f"DESCRIBE FUNCTION EXTENDED {CATALOG}.{SCHEMA}.grade").show(truncate=False)

## Permissions — `GRANT EXECUTE`

UC functions are governed objects. By default, only the owner can call them. To share:

```sql
GRANT EXECUTE ON FUNCTION main.default.grade TO `data-analysts`;
GRANT USAGE   ON SCHEMA   main.default       TO `data-analysts`;
GRANT USE CATALOG ON CATALOG main             TO `data-analysts`;
```

`USE CATALOG` + `USAGE` on the schema are needed too — otherwise the principal can't *see* the function to execute it.

## Limitations to know

- The body is **one expression**. For multi-step logic you'd express in code, use a Python UDF (notebook 14) or a view.
- No `LOOP`, no procedural blocks (Spark SQL isn't PL/SQL).
- Recursion isn't supported.
- A SQL UDF can only call other UC functions you have `EXECUTE` on.

## Cleanup

In [ ]:
for fn in ["grade", "fmt_money", "net_amount", "net_label", "top_scorers"]:
    spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.{fn}")
print("dropped")

## Takeaways

- SQL UDFs run inside Catalyst — no serialization cost, often as fast as a built-in.
- NULL propagates like in any SQL expression — no special handling.
- Define once in UC, share across the whole workspace via `GRANT EXECUTE`.
- `RETURNS TABLE` makes parameterized views.
- Body is a single SQL expression — no procedural code. Reach for Python UDFs (notebook 14) when that's a blocker.